### Testing RAG and Agents 🧪

### Setup

Install the project `requirements.txt`, restart the kernel, and run cells in order.
Ollama must be running with `qwen2.5:latest` available.
This lesson keeps the sample-based Ragas API used by the course; Ragas 0.4.3 emits deprecation warnings for it.


#### Create LLMs Instance with Local LLMs

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
from langchain_ollama import ChatOllama

# Works when the kernel starts in the project root or a section directory.
PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "Section9_Testing").is_dir() and (path / "requirements.txt").is_file()
)
load_dotenv(PROJECT_ROOT / ".env")

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen2.5:latest",
    temperature=0,
    num_predict=2048,
)


In [ ]:
from ragas import SingleTurnSample
from ragas.metrics import AspectCritic
from ragas.llms import LangchainLLMWrapper

test_data = {
    "user_input": "summarise given text\nThe company reported an 8% rise in Q3 2024, driven by strong performance in the Asian market. Sales in this region have significantly contributed to the overall growth. Analysts attribute this success to strategic marketing and product localization. The positive trend in the Asian market is expected to continue into the next quarter.",
    "response": "The company experienced an 8% increase in Q3 2024, largely due to effective marketing strategies and product adaptation, with expectations of continued growth in the coming quarter.",
}
evaluator_llm = LangchainLLMWrapper(llm)
metric = AspectCritic(name="summary_accuracy",llm=evaluator_llm, definition="Verify if the summary is accurate.")
await metric.single_turn_ascore(SingleTurnSample(**test_data))

### MultiTurnSample with RAGAS

In [ ]:
from ragas.messages import HumanMessage, AIMessage, ToolMessage, ToolCall

# User asks about the weather in New York City
user_message = HumanMessage(content="What's the weather like in New York City today?")

# AI decides to use a weather API tool to fetch the information
ai_initial_response = AIMessage(
    content="Let me check the current weather in New York City for you.",
    tool_calls=[ToolCall(name="WeatherAPI", args={"location": "New York City"})]
)

# Tool provides the weather information
tool_response = ToolMessage(content="It's sunny with a temperature of 75°F in New York City.")

# AI delivers the final response to the user
ai_final_response = AIMessage(content="It's sunny and 75 degrees Fahrenheit in New York City today.")

# Combine all messages into a list to represent the conversation
conversation = [
    user_message,
    ai_initial_response,
    tool_response,
    ai_final_response
]

In [ ]:
from ragas import MultiTurnSample
# Reference response for evaluation purposes
reference_response = "Provide the current weather in New York City to the user."


# Create the MultiTurnSample instance
sample = MultiTurnSample(
    user_input=conversation,
    reference=reference_response,
)

# This conversation needs a task-completion criterion, not summary accuracy.
weather_metric = AspectCritic(
    name="weather_task_completion",
    llm=evaluator_llm,
    definition="Verify that the assistant answers the user's weather question accurately using the weather tool result.",
)
await weather_metric.multi_turn_ascore(sample)